In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [10]:
import torch
import tiktoken
from utility import token_ids_to_text, text_to_token_ids
from model.architecture import generate_text_simple, GPTModel
from model.dummy_model import load_yaml

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
config = load_yaml("../model/config/model_config.yaml")["model"]
model = GPTModel(config)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=config["context_length"]
)
print(f"Output text: {token_ids_to_text(token_ids, tokenizer)}")

Output text: Every effort moves you investments cradle redirect WisdomeezCTardoooky ReynoldsLayout


## Calculating text generation loss

In [24]:
input1 = "every effort moves"
input2 = "I really like"
enc1 = torch.tensor(tokenizer.encode(input1))
enc2 = torch.tensor(tokenizer.encode(input2))
inputs = torch.stack((enc1, enc2))
print(inputs)
print(inputs.shape)

target1 = " effort moves you"
target2 = " really like chocolate"
t_enc1 = torch.tensor(tokenizer.encode(target1))
t_enc2 = torch.tensor(tokenizer.encode(target2))
targets = torch.stack((t_enc1, t_enc2))
print(targets)
print(targets.shape)

tensor([[16833,  3626,  6100],
        [   40,  1107,   588]])
torch.Size([2, 3])
tensor([[ 3626,  6100,   345],
        [ 1107,   588, 11311]])
torch.Size([2, 3])


In [26]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)
print(probas.shape)


torch.Size([2, 3, 50257])


In [27]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print(token_ids)
print(token_ids.shape)

tensor([[[38771],
         [33791],
         [42733]],

        [[ 2216],
         [20971],
         [47764]]])
torch.Size([2, 3, 1])


In [28]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")


Targets batch 1:  effort moves you
Outputs batch 1:  queens condom Waves
